# 🚀 Agent 工具（Agent Tools）

**欢迎来到 Kaggle 5 天 Agents 课程 Day 2！**

在 Day 1 中，你已经学会了如何创建可调用内置工具（如 Google Search）的 Agent，也学习了多智能体编排。现在我们进一步解锁 Agent 工具的完整能力：编写自定义逻辑、委派给专家 Agent，并处理真实场景中的复杂问题。

## 🤔 为什么 Agent 需要工具？

**问题**

没有工具时，Agent 的知识是“静态冻结”的：它无法访问今天的新闻，也无法读取你公司的库存系统。它与外部世界没有连接，因此无法真正替你执行动作。

**解决方案：** 工具可以把一个“孤立的 LLM”转变为真正能做事、能落地的 Agent。

在本 Notebook 中，你将：

- ✅ 把 Python 函数转换为 Agent 工具
- ✅ 构建一个 Agent，并让它**作为工具**被另一个 Agent 使用
- ✅ **搭建你的第一个多工具 Agent**
- ✅ 了解 ADK 中不同工具类型的适用场景

## ⚙️ 第 1 部分：环境准备

在进入今天的核心内容前，请按下面步骤完成环境配置。

### 1.1：安装依赖

Kaggle Notebooks 环境已经预装了 Python 版 [google-adk](https://google.github.io/adk-docs/) 及其所需依赖。

如果你要在课程外、自己的 Python 开发环境中安装并使用 ADK，可运行：

```
pip install google-adk
```

### 1.2：配置 Gemini API Key

本 Notebook 使用 [Gemini API](https://ai.google.dev/gemini-api/)，需要 API key。

**1. 获取 API key**

如果你还没有，可在 [Google AI Studio 创建 API key](https://aistudio.google.com/app/api-keys)。

**2. 将 key 添加到 Kaggle Secrets**

接下来，你需要把 API key 作为 Kaggle User Secret 添加到 Notebook 中。

1. 在 Notebook 编辑器顶部菜单选择 `Add-ons` -> `Secrets`。
2. 新建一个标签为 `GOOGLE_API_KEY` 的 secret。
3. 把 API key 粘贴到 "Value" 字段并点击 "Save"。
4. 确认 `GOOGLE_API_KEY` 左侧复选框已勾选，使其附加到该 Notebook。

**3. 在 Notebook 中鉴权**

运行下面代码单元，读取你保存的 `GOOGLE_API_KEY` 并设置为 Notebook 可用的环境变量：

In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


### 1.3：导入 ADK 组件

现在从 Agent Development Kit 与 Generative AI 库中导入所需组件。这样可以保持代码结构清晰，并确保我们具备所需构建块。

In [2]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4：辅助函数

下面的辅助函数用于打印代码执行工具生成的 Python 代码及其执行结果：

In [5]:
def show_python_code_and_result(response):
    for i in range(len(response)):
        # Check if the response contains a valid function call result from the code executor
        if (
            (response[i].content.parts)
            and (response[i].content.parts[0])
            and (response[i].content.parts[0].function_response)
            and (response[i].content.parts[0].function_response.response)
        ):
            response_code = response[i].content.parts[0].function_response.response
            if "result" in response_code and response_code["result"] != "```":
                if "tool_code" in response_code["result"]:
                    print(
                        "Generated Python Code >> ",
                        response_code["result"].replace("tool_code", ""),
                    )
                else:
                    print("Generated Python Response >> ", response_code["result"])


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.5：配置重试选项

在使用 LLM 时，你可能会遇到速率限制或服务短暂不可用等瞬时错误。重试选项会通过指数退避自动重试请求，以处理这类失败。

In [6]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

## 🤖 第 2 部分：什么是自定义工具（Custom Tools）？

**Custom Tools** 是你使用自己的代码与业务逻辑构建的工具。不同于 ADK 自带的内置工具，自定义工具能让你完全掌控功能行为。

**什么时候使用 Custom Tools？**

像 Google Search 这样的内置工具很强大，但**每个业务都有独特需求**，通用工具无法全部覆盖。Custom Tools 可以承载你的专属业务逻辑、连接内部系统并解决领域问题。ADK 提供了多种自定义工具类型来支持这些场景。

### 2.1：构建自定义 Function Tools

#### 示例：汇率转换 Agent

这个 Agent 可以把一种货币换算为另一种货币，并计算换汇手续费。它包含两个自定义工具，工作流如下：

1. **Fee Lookup Tool** - 查询该换汇方式的手续费（mock）
2. **Exchange Rate Tool** - 获取汇率（mock）
3. **Calculation Step** - 基于手续费与汇率计算最终换汇金额

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/currency-agent.png" width="600" alt="Currency Converter Agent">

### 🤔 2.2：如何定义一个 Tool？

**任何 Python 函数都可以成为 Agent 工具**，只需遵循这些简单规则：

1. 编写一个 Python 函数
2. 遵循下方最佳实践
3. 将函数加入 Agent 的 `tools=[]` 列表，其余交给 ADK 自动处理

#### 🏆 ADK 最佳实践示例

注意我们的工具遵循了 ADK 的最佳实践：

**1. 统一字典返回**：`{"status": "success", "data": ...}` 或 `{"status": "error", "error_message": ...}`  
**2. 清晰 Docstring**：LLM 会通过 docstring 理解何时、如何使用工具  
**3. Type Hints**：帮助 ADK 生成正确 schema（`str`、`dict` 等）  
**4. 错误处理**：结构化错误返回让 LLM 能更稳健地处理失败  


这些模式能让你的工具更可靠，也更容易被 LLM 正确使用。

👉 下面先看第一个工具：

In [7]:
# Pay attention to the docstring, type hints, and return value.
def get_fee_for_payment_method(method: str) -> dict:
    """Looks up the transaction fee percentage for a given payment method.

    This tool simulates looking up a company's internal fee structure based on
    the name of the payment method provided by the user.

    Args:
        method: The name of the payment method. It should be descriptive,
                e.g., "platinum credit card" or "bank transfer".

    Returns:
        Dictionary with status and fee information.
        Success: {"status": "success", "fee_percentage": 0.02}
        Error: {"status": "error", "error_message": "Payment method not found"}
    """
    # This simulates looking up a company's internal fee structure.
    fee_database = {
        "platinum credit card": 0.02,  # 2%
        "gold debit card": 0.035,  # 3.5%
        "bank transfer": 0.01,  # 1%
    }

    fee = fee_database.get(method.lower())
    if fee is not None:
        return {"status": "success", "fee_percentage": fee}
    else:
        return {
            "status": "error",
            "error_message": f"Payment method '{method}' not found",
        }


print("✅ Fee lookup function created")
print(f"💳 Test: {get_fee_for_payment_method('platinum credit card')}")

✅ Fee lookup function created
💳 Test: {'status': 'success', 'fee_percentage': 0.02}


接下来用同样的最佳实践定义第二个工具 `get_exchange_rate`。

In [8]:
def get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    """Looks up and returns the exchange rate between two currencies.

    Args:
        base_currency: The ISO 4217 currency code of the currency you
                       are converting from (e.g., "USD").
        target_currency: The ISO 4217 currency code of the currency you
                         are converting to (e.g., "EUR").

    Returns:
        Dictionary with status and rate information.
        Success: {"status": "success", "rate": 0.93}
        Error: {"status": "error", "error_message": "Unsupported currency pair"}
    """

    # Static data simulating a live exchange rate API
    # In production, this would call something like: requests.get("api.exchangerates.com")
    rate_database = {
        "usd": {
            "eur": 0.93,  # Euro
            "jpy": 157.50,  # Japanese Yen
            "inr": 83.58,  # Indian Rupee
        }
    }

    # Input validation and processing
    base = base_currency.lower()
    target = target_currency.lower()

    # Return structured result with status
    rate = rate_database.get(base, {}).get(target)
    if rate is not None:
        return {"status": "success", "rate": rate}
    else:
        return {
            "status": "error",
            "error_message": f"Unsupported currency pair: {base_currency}/{target_currency}",
        }


print("✅ Exchange rate function created")
print(f"💱 Test: {get_exchange_rate('USD', 'EUR')}")

✅ Exchange rate function created
💱 Test: {'status': 'success', 'rate': 0.93}


现在创建我们的货币换算 Agent。请特别注意 instruction 是如何引用工具的：

**关键点：**
- `tools=[]` 列表告诉 Agent 它可以调用哪些函数
- instruction 通过函数名精确引用工具（例如 `get_fee_for_payment_method()`）
- Agent 会基于这些名称判断何时、如何调用每个工具

In [9]:
# Currency agent with custom function tools
currency_agent = LlmAgent(
    name="currency_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a smart currency conversion assistant.

    For currency conversion requests:
    1. Use `get_fee_for_payment_method()` to find transaction fees
    2. Use `get_exchange_rate()` to get currency conversion rates
    3. Check the "status" field in each tool's response for errors
    4. Calculate the final amount after fees based on the output from `get_fee_for_payment_method` and `get_exchange_rate` methods and provide a clear breakdown.
    5. First, state the final converted amount.
        Then, explain how you got that result by showing the intermediate amounts. Your explanation must include: the fee percentage and its
        value in the original currency, the amount remaining after the fee, and the exchange rate used for the final conversion.

    If any tool returns status "error", explain the issue to the user clearly.
    """,
    tools=[get_fee_for_payment_method, get_exchange_rate],
)

print("✅ Currency agent created with custom function tools")
print("🔧 Available tools:")
print("  • get_fee_for_payment_method - Looks up company fee structure")
print("  • get_exchange_rate - Gets current exchange rates")

✅ Currency agent created with custom function tools
🔧 Available tools:
  • get_fee_for_payment_method - Looks up company fee structure
  • get_exchange_rate - Gets current exchange rates


In [10]:
# Test the currency agent
currency_runner = InMemoryRunner(agent=currency_agent)
_ = await currency_runner.run_debug(
    "I want to convert 500 US Dollars to Euros using my Platinum Credit Card. How much will I receive?"
)


 ### Created new session: debug_session_id

User > I want to convert 500 US Dollars to Euros using my Platinum Credit Card. How much will I receive?


currency_agent > You will receive 455.70 Euros.

This is calculated as follows:
The fee for using a Platinum Credit Card is 2%, which is 10.00 USD.
After deducting the fee, the amount is 490.00 USD.
The exchange rate from USD to EUR is 0.93.
So, 490.00 USD is converted to 455.70 EUR.


**非常好！** 现在我们的 Agent 已经可以使用结构化返回的自定义业务逻辑。

## 💻 第 3 部分：用代码提升 Agent 可靠性

当前 Agent 的 instruction 是“*计算扣费后的最终金额*”，但 LLM 在数学计算上并不总是稳定，可能会算错或使用不一致公式。

##### 💡 **解决方案：** 让 Agent 先生成 Python 代码，再执行代码得到最终结果！相比让 LLM“心算”，代码执行更可靠。

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/enhanced-currency-agent.png" width="800" alt="Enhanced Currency Converter Agent">

### 3.1：内置 Code Executor

ADK 提供了可在沙箱中运行代码的内置 Code Executor。**注意：** 该能力依赖 Gemini 的 Code Execution。

我们来创建一个 `calculation_agent`，它接收 Python 代码并通过 `BuiltInCodeExecutor` 执行。

In [12]:
calculation_agent = LlmAgent(
    name="CalculationAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""You are a specialized calculator that ONLY responds with Python code. You are forbidden from providing any text, explanations, or conversational responses.
 
     Your task is to take a request for a calculation and translate it into a single block of Python code that calculates the answer.
     
     **RULES:**
    1.  Your output MUST be ONLY a Python code block.
    2.  Do NOT write any text before or after the code block.
    3.  The Python code MUST calculate the result.
    4.  The Python code MUST print the final result to stdout.
    5.  You are PROHIBITED from performing the calculation yourself. Your only job is to generate the code that will perform the calculation.
   
    Failure to follow these rules will result in an error.
       """,
    code_executor=BuiltInCodeExecutor(),  # Use the built-in Code Executor Tool. This gives the agent code execution capabilities
)

### 3.2：更新 Agent 的 instruction 与工具集

我们要做两件关键事情：

1. **更新 `currency_agent` 的 instruction，让它生成 Python 代码**
- 原版："*Calculate the final amount after fees*"（数学指令较模糊）
- 增强版："*Generate a Python code ... and use the `calculation_agent` to run the code and compute final amount*"

2. **把 `calculation_agent` 加入工具集**

    ADK 允许通过 `AgentTool` 把任意 Agent 当作工具使用。

- 在 tools 中加入 `AgentTool(agent=calculation_agent)`
- 该专家 Agent 会以“可调用工具”形式暴露给根 Agent

下面看实际效果：

In [13]:
enhanced_currency_agent = LlmAgent(
    name="enhanced_currency_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    # Updated instruction
    instruction="""You are a smart currency conversion assistant. You must strictly follow these steps and use the available tools.

  For any currency conversion request:

   1. Get Transaction Fee: Use the get_fee_for_payment_method() tool to determine the transaction fee.
   2. Get Exchange Rate: Use the get_exchange_rate() tool to get the currency conversion rate.
   3. Error Check: After each tool call, you must check the "status" field in the response. If the status is "error", you must stop and clearly explain the issue to the user.
   4. Calculate Final Amount (CRITICAL): You are strictly prohibited from performing any arithmetic calculations yourself. You must use the calculation_agent tool to generate Python code that calculates the final converted amount. This 
      code will use the fee information from step 1 and the exchange rate from step 2.
   5. Provide Detailed Breakdown: In your summary, you must:
       * State the final converted amount.
       * Explain how the result was calculated, including:
           * The fee percentage and the fee amount in the original currency.
           * The amount remaining after deducting the fee.
           * The exchange rate applied.
    """,
    tools=[
        get_fee_for_payment_method,
        get_exchange_rate,
        AgentTool(agent=calculation_agent),  # Using another agent as a tool!
    ],
)

print("✅ Enhanced currency agent created")
print("🎯 New capability: Delegates calculations to specialist agent")
print("🔧 Tool types used:")
print("  • Function Tools (fees, rates)")
print("  • Agent Tool (calculation specialist)")

✅ Enhanced currency agent created
🎯 New capability: Delegates calculations to specialist agent
🔧 Tool types used:
  • Function Tools (fees, rates)
  • Agent Tool (calculation specialist)


In [14]:
# Define a runner
enhanced_runner = InMemoryRunner(agent=enhanced_currency_agent)

In [15]:
# Test the enhanced agent
response = await enhanced_runner.run_debug(
    "Convert 1,250 USD to INR using a Bank Transfer. Show me the precise calculation."
)


 ### Created new session: debug_session_id

User > Convert 1,250 USD to INR using a Bank Transfer. Show me the precise calculation.


enhanced_currency_agent > Here's a breakdown of your currency conversion:

*   **Transaction Fee:** A fee of 1% was applied for the Bank Transfer, amounting to 12.50 USD.
*   **Amount After Fee:** After deducting the fee, 1237.50 USD remained.
*   **Exchange Rate:** The exchange rate used was 1 USD to 83.58 INR.
*   **Final Converted Amount:** You will receive 103507.50 INR.


**很棒！** 注意发生了什么：

- 当 Currency agent 调用 `CalculationAgent` 时，会传入生成的 Python 代码
- `CalculationAgent` 再通过 `BuiltInCodeExecutor` 执行代码，给出精确计算结果，而不是依赖 LLM 估算

现在你可以用本 Notebook 前面定义的辅助函数，检查响应中“生成的 Python 代码”以及“代码执行结果”部分：

In [16]:
show_python_code_and_result(response)

Generated Python Code >>  
import decimal

def calculate_final_amount(amount_usd, fee_percentage, exchange_rate):
    fee_amount_usd = amount_usd * fee_percentage
    amount_after_fee_usd = amount_usd - fee_amount_usd
    converted_amount_inr = amount_after_fee_usd * exchange_rate
    return {
        "fee_amount_usd": fee_amount_usd,
        "amount_after_fee_usd": amount_after_fee_usd,
        "converted_amount_inr": converted_amount_inr
    }

# Given values
amount_usd = 1250
fee_percentage = 0.01
exchange_rate = 83.58

# Perform the calculation
result = calculate_final_amount(decimal.Decimal(amount_usd), decimal.Decimal(fee_percentage), decimal.Decimal(exchange_rate))

print(f"Fee Amount (USD): {result['fee_amount_usd']:.2f}")
print(f"Amount after fee (USD): {result['amount_after_fee_usd']:.2f}")
print(f"Converted Amount (INR): {result['converted_amount_inr']:.2f}")



### 🤔 3.3：Agent Tools 与 Sub-Agents 有什么区别？

这是一个非常常见的问题！两者都涉及多个 Agent，但工作方式完全不同：

**Agent Tools（我们当前用的是这个）：**
- Agent A 把 Agent B 作为工具调用
- Agent B 的结果会**返回给 Agent A**
- Agent A 继续掌控流程并维持对话
- **适用场景**：针对特定任务做能力委派（例如计算）

**Sub-Agents（另一种模式）：**
- Agent A 会把控制权**完整移交给 Agent B**
- Agent B 接手后续所有用户输入
- Agent A 退出主流程
- **适用场景**：面向专家分层转接（例如客服分级）

**在我们的换汇示例中：** 我们希望 currency agent 拿到计算结果后继续处理，所以使用 **Agent Tools**，而不是 sub-agents。

## 🧰 第 4 部分：ADK 工具类型完整指南

你已经看过工具如何实际工作，现在我们来系统理解 ADK 的完整工具箱：

大体分为两类：**自定义工具（Custom Tools）** 与 **内置工具（Built-in Tools）**

### **1. 自定义工具（Custom Tools）**

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/custom-tools.png" width="800" alt="Custom Tools">

**是什么**：你为特定需求自行构建的工具

**优势**：对功能有完全控制权，可以精确实现 Agent 需要的能力

#### **Function Tools** ✅（你已经用过）
- **是什么**：把 Python 函数转换成 Agent 工具
- **示例**：`get_fee_for_payment_method`、`get_exchange_rate`
- **优势**：几乎可即时把任意 Python 函数工具化

#### **Long Running Function Tools**
- **是什么**：用于耗时较长操作的函数工具
- **示例**：人审审批、文件处理
- **优势**：Agent 可在等待期间继续处理其他任务

#### **Agent Tools** ✅（你已经用过）
- **是什么**：把其他 Agent 当工具来调用
- **示例**：`AgentTool(agent=calculation_agent)`
- **优势**：可构建专家 Agent 并在不同系统中复用

#### **MCP Tools**
- **是什么**：来自 Model Context Protocol 服务器的工具
- **示例**：文件系统访问、Google Maps、数据库
- **优势**：无需手写复杂集成，即可连接 MCP 兼容服务

#### **OpenAPI Tools**
- **是什么**：由 API 规范自动生成的工具
- **示例**：将 REST API 端点直接变为可调用工具
- **优势**：无需手写代码，提供 API spec 即可得到可用工具

### **2. 内置工具（Built-in Tools）**

<img src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day2/built-in-tools.png" width="1200" alt="Built-in Tools">

**是什么**：ADK 提供的预构建工具

**优势**：零开发成本，开箱即用

#### **Gemini Tools** ✅（你已经用过）
- **是什么**：利用 Gemini 能力的工具
- **示例**：`google_search`、`BuiltInCodeExecutor`
- **优势**：稳定、经过验证，可直接投入使用

#### **Google Cloud Tools** [需要 Google Cloud 访问权限]
- **是什么**：用于 Google Cloud 服务和企业级集成的工具
- **示例**：`BigQueryToolset`、`SpannerToolset`、`APIHubToolset`
- **优势**：提供企业级数据库/API 接入，并内置安全机制

#### **第三方工具（Third-party Tools）**
- **是什么**：对现有工具生态的封装
- **示例**：Hugging Face、Firecrawl、GitHub Tools
- **优势**：复用既有工具投资，无需重复造轮子

## ✅ 恭喜！

你已经掌握了如何构建“会行动”的 Agent，不再只是文本回复。在本 Notebook 中，你学习了：

1. 🔧 **Function Tools** - 将 Python 函数转换为 Agent 工具
2. 🤖 **Agent Tools** - 构建专家 Agent 并把它们作为工具复用
3. 🧰 **完整工具箱** - 理解 ADK 各类工具及其适用场景

**ℹ️ 说明：无需提交！**

这个 Notebook 仅用于动手实践与学习。完成课程**不需要**把它提交到任何地方。

### 📚 继续学习

你可以参考以下文档深入了解：

- [ADK 文档总览](https://google.github.io/adk-docs/)
- [ADK Tools 文档](https://google.github.io/adk-docs/tools/)
- [ADK 自定义工具指南](https://google.github.io/adk-docs/tools-custom/)
- [ADK Function Tools](https://google.github.io/adk-docs/tools/function-tools/)
- [ADK Plugins 概览](https://google.github.io/adk-docs/plugins/)

### 🎯 下一步

你已经打下 Agent 工具能力的基础。

准备迎接下一个挑战了吗？继续学习下一本 Notebook，深入 **tool patterns**！